# Smoke tests for `geolab-base`

For every package in `environment.yml` (conda + pip): try to import it and
exercise one minimal API call. CLI-only packages get a `which`/`--version`
check instead. A failure here means something installed but doesn't load,
which is usually a sign of an ABI mismatch or a missing system library.

Run all cells. The summary at the bottom lists pass/fail per package.

## Setup

In [ ]:
import importlib
import shutil
import subprocess
import sys

RESULTS = []


def py(modname, alias=None, smoke=None):
    """Import `modname` and optionally run `smoke(mod)` as a sanity check."""
    label = alias or modname
    try:
        mod = importlib.import_module(modname)
        if smoke is not None:
            smoke(mod)
        version = getattr(mod, '__version__', '')
        RESULTS.append((label, 'OK', str(version), ''))
    except Exception as exc:
        RESULTS.append((label, 'FAIL', '', f'{type(exc).__name__}: {exc}'))


def cli(cmd, version_flag='--version'):
    """Verify `cmd` is on $PATH and responds to a version flag."""
    path = shutil.which(cmd)
    if not path:
        RESULTS.append((cmd, 'FAIL', '', 'not on $PATH'))
        return
    try:
        r = subprocess.run([cmd, version_flag],
                           capture_output=True, text=True, timeout=10)
        line = (r.stdout or r.stderr).strip().splitlines()
        version = line[0] if line else 'on PATH'
        RESULTS.append((cmd, 'OK', version[:80], ''))
    except Exception as exc:
        RESULTS.append((cmd, 'OK', 'on PATH', f'{type(exc).__name__}'))


print(f'Python {sys.version}')
print(f'sys.prefix: {sys.prefix}')

## Cloud & storage

In [ ]:
cli('aws')
py('awswrangler')
py('boto3', smoke=lambda m: m.client('s3', region_name='us-east-1'))
py('fsspec', smoke=lambda m: m.filesystem('memory'))
py('obstore')
py('s3fs', smoke=lambda m: m.S3FileSystem)

## Geospatial

In [ ]:
py('cartopy.crs', alias='cartopy',
   smoke=lambda m: m.PlateCarree())
py('contextily')
py('fiona', smoke=lambda m: m.supported_drivers)
py('folium',
   smoke=lambda m: m.Map(location=[0, 0], zoom_start=2))
py('osgeo.gdal', alias='gdal',
   smoke=lambda m: m.VersionInfo('RELEASE_NAME'))
py('ipyleaflet', smoke=lambda m: m.Map())
py('lonboard')
py('pyproj', smoke=lambda m: m.CRS('EPSG:4326'))
py('shapely.geometry', alias='shapely',
   smoke=lambda m: m.Point(0, 0).buffer(1).area)

## Core scientific stack

In [ ]:
py('numpy', smoke=lambda m: int(m.array([1, 2, 3]).sum()))
py('numba', smoke=lambda m: m.njit(lambda x: x + 1)(1))
py('scipy.stats', alias='scipy', smoke=lambda m: m.norm.cdf(0))
py('pandas',
   smoke=lambda m: m.DataFrame({'a': [1, 2]}).shape)
py('geopandas')
import matplotlib; matplotlib.use('Agg')
py('matplotlib', alias='matplotlib-base',
   smoke=lambda m: m.figure.Figure())
py('xarray',
   smoke=lambda m: m.DataArray([1, 2, 3]).sum().item())
py('netCDF4', alias='netcdf4')
py('h5py')
py('h5netcdf')
py('pyarrow',
   smoke=lambda m: m.array([1, 2, 3]).to_pylist())
py('zarr',
   smoke=lambda m: m.zeros((3,), chunks=3, dtype='f4'))
py('virtualizarr')
py('bottleneck',
   smoke=lambda m: m.nansum([1.0, 2.0, float('nan'), 3.0]))
py('flox')
py('pooch')
py('dask.array', alias='dask',
   smoke=lambda m: m.ones(10, chunks=5).sum().compute())
py('distributed')
py('dask_gateway', alias='dask-gateway')
py('cvxpy', smoke=lambda m: m.Variable(name='x'))

## Geo / geoscience

In [ ]:
py('dascore')
cli('gmt', version_flag='--version')
py('obspy',
   smoke=lambda m: m.UTCDateTime('2020-01-01').timestamp)
py('obsplus')
py('pygmt')

## geolab-gmt additions

In [ ]:
import tempfile
from pathlib import Path


def _taup_smoke():
    """Compute a real P travel time with the bundled iasp91 model."""
    label = 'taup'
    path = shutil.which('taup')
    if not path:
        RESULTS.append((label, 'FAIL', '', 'not on $PATH'))
        return
    try:
        r = subprocess.run(
            [path, 'time', '-h', '50', '--deg', '60', '--ph', 'P'],
            capture_output=True, text=True, timeout=30, check=True,
        )
        if 'iasp91' not in r.stdout.lower() or ' P ' not in r.stdout:
            raise RuntimeError(f'unexpected output: {r.stdout!r}')
        RESULTS.append((label, 'OK', 'computed P travel time (iasp91)', ''))
    except Exception as exc:
        RESULTS.append((label, 'FAIL', '', f'{type(exc).__name__}: {exc}'))


def _gmt_compute_smoke():
    """Render a real coastline map, not just check `gmt --version`."""
    label = 'gmt (compute)'
    try:
        with tempfile.TemporaryDirectory() as tmp:
            def run(*args):
                subprocess.run(args, cwd=tmp, check=True,
                                capture_output=True, text=True, timeout=60)

            run('gmt', 'begin', 'testmap', 'png')
            run('gmt', 'coast', '-R0/10/0/10', '-JM10c', '-Ba', '-Ggray')
            run('gmt', 'end')

            out = Path(tmp) / 'testmap.png'
            if not out.exists() or out.stat().st_size == 0:
                raise RuntimeError('gmt did not produce testmap.png')

            # `gmt end` doesn't reliably remove this session's on-disk
            # bookkeeping (~/.gmt/sessions/<name>), and later GMT/PyGMT
            # calls in this same $HOME can read that leftover state and
            # decide they're in classic mode. Clear it immediately so this
            # CLI-based test never pollutes anything that runs after it.
            run('gmt', 'clear', 'sessions')
        RESULTS.append((label, 'OK', 'rendered coastline map', ''))
    except Exception as exc:
        RESULTS.append((label, 'FAIL', '', f'{type(exc).__name__}: {exc}'))


_PYGMT_FIGURE_SCRIPT = """
import sys
import tempfile
from pathlib import Path

import pygmt

with tempfile.TemporaryDirectory() as tmp:
    fig = pygmt.Figure()
    fig.basemap(region=[0, 10, 0, 10], projection="X10c/10c", frame=True)
    fig.coast(shorelines=True)
    out = Path(tmp) / "test.png"
    fig.savefig(str(out))
    if not (out.exists() and out.stat().st_size > 0):
        sys.exit("figure was not written")
"""


def _pygmt_figure_smoke(m):
    """Actually render a figure through GMT's C library, not just import it.

    PyGMT's Figure API only exists in modern mode -- it never uses classic
    mode. But GMT's modern-mode bookkeeping is process-global state under
    ~/.gmt/sessions/<name>, so it can be corrupted by anything else that
    touched the same $HOME -- e.g. _gmt_compute_smoke's CLI `gmt begin`/
    `end` (now cleaned up there directly), a container reusing a low PID
    across kernel restarts, or a future test added to this notebook.
    Running in a fresh subprocess is defense in depth against the
    process-local part of that; it doesn't by itself fix on-disk pollution
    under a shared $HOME, which is why _gmt_compute_smoke also clears its
    own session now.
    """
    r = subprocess.run([sys.executable, '-c', _PYGMT_FIGURE_SCRIPT],
                        capture_output=True, text=True, timeout=60)
    if r.returncode != 0:
        raise RuntimeError((r.stderr or r.stdout).strip()[-2000:])


def _gdal_jp2_smoke(m):
    """Write and read back a real JP2 file, not just check the driver exists."""
    import numpy as np

    driver = m.GetDriverByName('JP2OpenJPEG')
    assert driver is not None, 'JP2OpenJPEG driver not registered'

    # Use a size representative of real rasters (64x64), not 8x8: JP2's
    # default RESOLUTIONS picks enough wavelet decomposition levels that the
    # smallest overview stays <=128x128, which is degenerate/edge-case for a
    # tiny test image and not how any real caller would use this driver.
    size = 64
    mem_ds = m.GetDriverByName('MEM').Create('', size, size, 1, m.GDT_Byte)
    data = (np.arange(size * size, dtype='uint32') % 256).astype('uint8').reshape(size, size)
    mem_ds.GetRasterBand(1).WriteArray(data)

    with tempfile.TemporaryDirectory() as tmp:
        out_path = str(Path(tmp) / 'test.jp2')
        # This driver is used for cartographic output (linework, symbology,
        # categorical boundaries), not remote-sensing imagery -- lossy
        # quantization isn't acceptable here, so all three creation options
        # GDAL documents as required for a lossless round-trip must be set:
        # REVERSIBLE=YES, QUALITY=100, and YCBCR420=NO (the current test
        # data is single-band so YCBCR420 is moot, but pin it explicitly
        # since real cartographic tiles are RGB and its default would
        # subsample chroma).
        # https://gdal.org/en/stable/drivers/raster/jp2openjpeg.html
        jp2_ds = driver.CreateCopy(
            out_path, mem_ds,
            options=['REVERSIBLE=YES', 'QUALITY=100', 'YCBCR420=NO'])
        jp2_ds = None
        mem_ds = None

        check_ds = m.Open(out_path)
        assert check_ds is not None, 'failed to reopen JP2 file'
        result = check_ds.GetRasterBand(1).ReadAsArray()
        assert (result == data).all(), 'pixel data did not round-trip losslessly'


cli('nano')
_taup_smoke()
_gmt_compute_smoke()
py('pygmt', alias='pygmt (figure)', smoke=_pygmt_figure_smoke)
py('osgeo.gdal', alias='gdal (JP2OpenJPEG)', smoke=_gdal_jp2_smoke)

## Utilities

In [ ]:
py('tqdm',
   smoke=lambda m: list(m.tqdm(range(3), disable=True)))
py('requests')
py('yaml', alias='pyyaml',
   smoke=lambda m: m.safe_load('a: 1'))
cli('gs', version_flag='--version')      # ghostscript
cli('ffmpeg', version_flag='-version')

## Dev tools

In [ ]:
cli('gh')
cli('gh-scoped-creds')
py('pytest')
cli('ruff')

## Jupyter stack & extensions

In [ ]:
py('jupyterhub')
py('jupyter_server')
py('jupyterlab')
py('ipykernel')
py('jupyter_resource_usage', alias='jupyter-resource-usage')
py('jupyter_ruff', alias='jupyter-ruff')
py('jupyter_server_proxy', alias='jupyter-server-proxy')
py('jupyterlab_git', alias='jupyterlab-git')
py('jupyterlab_myst', alias='jupyterlab-myst')
py('jupyterlab_code_formatter')
py('jupyterlab_pygments')
py('nbdime')

## pip packages & visualization

In [ ]:
# EarthScope --------------------------------------------------
py('earthscope_sdk', alias='earthscope-sdk')
cli('es')                       # earthscope-cli entry point
py('earthscopestraintools')

# Jupyter add-ons ---------------------------------------------
py('jupyterlab_jupyterbook_navigation')

# Visualization & data frames ---------------------------------
py('altair',
   smoke=lambda m: m.Chart())
py('plotly')
py('polars',
   smoke=lambda m: m.DataFrame({'a': [1, 2]}))
py('vegafusion')
py('vl_convert', alias='vl-convert-python')
py('ipympl')
py('hvplot')
py('holoviews', alias='holoviews',
   smoke=lambda m: m.Curve([1, 2, 3]))
py('panel')

## Interactive widgets

In [ ]:
py('ipywidgets',
   smoke=lambda m: m.IntSlider(value=5, min=0, max=10))
py('anywidget')
py('bqplot')
py('ipytree', smoke=lambda m: m.Node(name='root'))
py('itables')
py('ipydatagrid')
from sidecar import Sidecar  # noqa: F401
py('sidecar')

## Summary

In [ ]:
import pandas as pd
from IPython.display import display

df = pd.DataFrame(RESULTS,
                  columns=['package', 'status', 'version', 'error'])

passed = int((df['status'] == 'OK').sum())
total = len(df)
failed = total - passed

print(f'Results: {passed}/{total} OK, {failed} failed')
if failed:
    print('\nFailures:')
    for _, row in df[df['status'] == 'FAIL'].iterrows():
        print(f"  {row['package']:35s}  {row['error']}")

df.style.map(
    lambda v: ('color: red; font-weight: bold' if v == 'FAIL'
               else 'color: green'),
    subset=['status']
)